# 🔢 Day 5A — Embeddings & Semantic Similarity
**Agentic AI for Tax Technologists · 12-Day Program**

---

## 📖 The Story So Far

| Day | Built | Gap closed |
|-----|-------|------------|
| Day 2 | ReAct agent — tool calls + memory | — |
| Day 3 | CoT + Pydantic — structured outputs | Uncontrolled reasoning |
| Day 4 | Function calling — extraction from messy docs | Clean inputs only |
| **Day 5** | **RAG — agent retrieves docs automatically** | **Manual document routing** |

**Day 5 has two parts:**
- **5A (this notebook):** Understand embeddings hands-on — *before* touching a vector store
- **5B:** Build the vector store with ChromaDB
- **5C:** Wire retrieval into generation — the full RAG pipeline

**Why start with embeddings?** A vector store is a database of embeddings. If you don't understand what an embedding *is*, the vector store feels like magic. After this notebook, it won't.

---

## 🎯 Learning Objectives
1. Call the Azure OpenAI embeddings API and inspect the output
2. Understand what a 1536-dimensional vector represents
3. Compute cosine similarity between tax phrases
4. Confirm that semantically similar phrases cluster together — not just keyword matches
5. Visualise high-dimensional vectors in 2D with PCA

---
## ⏱️ Time: 30 minutes

---
## ⚙️ Step 1: Install & Import

In [ ]:
%pip install openai numpy scikit-learn matplotlib --quiet

In [ ]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from getpass import getpass
from openai import AzureOpenAI

print("✅ Imports OK")

---
## 🔑 Step 2: Configure Azure OpenAI

> **Note on deployments:** Embeddings use a *different* deployment from your chat model.
> You need a deployment of `text-embedding-ada-002` or `text-embedding-3-small`.
> Ask your facilitator for the embeddings deployment name.

In [ ]:
AZURE_OPENAI_ENDPOINT       = input("Endpoint (e.g. https://xxx.openai.azure.com/): ").strip()
AZURE_OPENAI_API_KEY        = getpass("API Key: ")
AZURE_DEPLOYMENT_NAME       = input("Chat deployment name (e.g. gpt-4o): ").strip()
AZURE_EMBEDDING_DEPLOYMENT  = input("Embeddings deployment name (e.g. text-embedding-ada-002): ").strip()
AZURE_API_VERSION           = "2024-08-01-preview"

client = AzureOpenAI(
    azure_endpoint = AZURE_OPENAI_ENDPOINT,
    api_key        = AZURE_OPENAI_API_KEY,
    api_version    = AZURE_API_VERSION,
)
print("✅ Azure OpenAI client initialised successfully!")

---
## 🔢 Step 3: Your First Embedding

In [ ]:
def get_embedding(text):
    """Call Azure OpenAI embeddings API. Returns a list of floats."""
    response = client.embeddings.create(
        model = AZURE_EMBEDDING_DEPLOYMENT,
        input = text
    )
    return response.data[0].embedding


# Get the embedding for a single tax phrase
phrase = "GST rate on export of IT services"
vec = get_embedding(phrase)

print(f"Input text   : {phrase}")
print(f"Vector length: {len(vec)} dimensions")
print(f"First 8 values: {[round(v, 4) for v in vec[:8]]}")
print(f"Value range  : {min(vec):.4f} to {max(vec):.4f}")
print()
print("Each number encodes a dimension of meaning.")
print("Texts with similar meaning will have similar numbers in similar positions.")

---
## 📐 Step 4: Cosine Similarity Helper

In [ ]:
def cosine_similarity(vec_a, vec_b):
    """Compute cosine similarity between two vectors. Returns float 0.0–1.0."""
    a = np.array(vec_a)
    b = np.array(vec_b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


# Quick sanity check: a phrase should be identical to itself
v1 = get_embedding("GST on exports")
v2 = get_embedding("GST on exports")
print(f"Identical phrases similarity: {cosine_similarity(v1, v2):.4f}  (should be ~1.00)")

---
## 🧪 Step 5: Semantic Similarity on Tax Phrases

We'll test 8 tax phrases. The hypothesis:
- Phrases about export of services should cluster together
- Phrases about TDS on contractors should cluster together
- The two clusters should be far apart

**This is why RAG works** — a question about exports retrieves export-related chunks, not TDS chunks.

In [ ]:
TAX_PHRASES = [
    # Export / zero-rated cluster
    "GST rate on export of IT services to USA",
    "zero-rated supply under IGST Act Section 16",
    "LUT filing for exporters of services",
    "place of supply for export of services",
    # TDS / contractor cluster
    "TDS rate on contractor payments Section 194C",
    "TDS deduction on sub-contractor payment",
    "Form 26Q for contractor TDS returns",
    "advance tax instalments due dates",
]

print("Computing embeddings for 8 tax phrases...")
embeddings = {phrase: get_embedding(phrase) for phrase in TAX_PHRASES}
print("✅ Done")

In [ ]:
# Compute full pairwise similarity matrix
n = len(TAX_PHRASES)
short_labels = [
    "export IT services", "zero-rated IGST S.16", "LUT filing", "place of supply",
    "TDS 194C contractors", "TDS sub-contractor", "Form 26Q", "advance tax dates"
]

sim_matrix = np.zeros((n, n))
for i, p1 in enumerate(TAX_PHRASES):
    for j, p2 in enumerate(TAX_PHRASES):
        sim_matrix[i][j] = cosine_similarity(embeddings[p1], embeddings[p2])

print("COSINE SIMILARITY MATRIX")
print(f"{'':25}" + "".join(f"{l[:14]:>16}" for l in short_labels[:4]) + "  ...")
for i, label in enumerate(short_labels):
    row = "".join(f"{sim_matrix[i][j]:>16.3f}" for j in range(4))
    print(f"  {label[:23]:<23}  {row}")

print()
# Key comparisons
comparisons = [
    ("export IT services", "zero-rated IGST S.16", 0, 1),
    ("export IT services", "LUT filing", 0, 2),
    ("export IT services", "TDS 194C contractors", 0, 4),
    ("export IT services", "advance tax dates", 0, 7),
]
print("KEY COMPARISONS")
print("-" * 65)
for a, b, i, j in comparisons:
    score = sim_matrix[i][j]
    bar = "█" * int(score * 20)
    close = "← SIMILAR" if score > 0.75 else ("← DISTANT" if score < 0.5 else "")
    print(f"  {a:<22} ↔ {b:<22}  {score:.3f}  {bar:<14} {close}")

---
## 📊 Step 6: Visualise in 2D with PCA

In [ ]:
from sklearn.decomposition import PCA

vectors = np.array([embeddings[p] for p in TAX_PHRASES])

# Reduce 1536 dimensions → 2 for plotting
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(vectors)

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor("#F4F7FB")
ax.set_facecolor("white")

colors  = ["#4F46E5"] * 4 + ["#F5A623"] * 3 + ["#E05C5C"] * 1
markers = ["o"] * 4 + ["s"] * 3 + ["^"] * 1

for i, (x, y) in enumerate(coords):
    ax.scatter(x, y, c=colors[i], marker=markers[i], s=120, zorder=5)
    ax.annotate(
        short_labels[i],
        (x, y), textcoords="offset points", xytext=(8, 4),
        fontsize=9, color="#334E68"
    )

# Legend
from matplotlib.patches import Patch
ax.legend(handles=[
    Patch(facecolor="#4F46E5", label="Export / zero-rated cluster"),
    Patch(facecolor="#F5A623", label="TDS / contractor cluster"),
    Patch(facecolor="#E05C5C", label="Other"),
], fontsize=9, loc="upper right")

ax.set_title("Tax phrase embeddings projected to 2D (PCA)", fontsize=12, fontweight="bold", color="#0F1F3D")
ax.set_xlabel("PCA dimension 1", fontsize=10, color="#64748B")
ax.set_ylabel("PCA dimension 2", fontsize=10, color="#64748B")
ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("day5a_embeddings_pca.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Plot saved to day5a_embeddings_pca.png")
print()
print("Observation: phrases in the same tax cluster land close together.")
print("This is exactly how ChromaDB finds relevant chunks — by finding nearby vectors.")

---
## 🎓 Key Insight

```
Keyword search:  'export of services' → only matches documents containing those exact words

Embedding search: 'export of services' → matches 'zero-rated supply', 'LUT filing',
                  'place of supply IGST' — even though different words
```

For a tax knowledge base with 300+ circulars using varied terminology, this is **the** difference between finding the right section and missing it entirely.

---
## ➡️ Next: Notebook 5B — Build the Vector Store

We take these embeddings and store them in ChromaDB alongside their source metadata — building the knowledge base the RAG pipeline will query.